# Suite TRN — Translators

Every pair of dialects has its own bidirectional translator (`mbse.Expressions.Translators`), declared as rules
that pair two patterns with holes. Translating co-traverses each rule's pattern with the expression to bind the
holes, then instantiates the other side, so every node is translated once and sharing is kept. A dialect with no
binding gets a let's value substituted for its name, and imports are dropped where the other side cannot declare
them, or added where the translation needs them (`import numpy as np`). Translations keep an expression's form; its value follows
each dialect's rules, and this suite shows where those agree and where they do not.

In [ ]:
import itertools

import numpy as np

from mbse.Expressions import Domains as D, Evaluators as V, Expressions as E, Translators
from mbse.Expressions.Dialects.Ccpp import Evaluators as CV, Expressions as C
from mbse.Expressions.Dialects.Excel import Evaluators as XV, Expressions as X
from mbse.Expressions.Dialects.Latex import Expressions as L
from mbse.Expressions.Dialects.Matlab import Evaluators as MV, Expressions as M
from mbse.Expressions.Dialects.Python import Evaluators as PV, Expressions as P
from mbse.Expressions.Dialects.SystemVerilog import Evaluators as SVV, Expressions as SV
from mbse.Expressions.Framework import Terms as F
from mbse.Expressions.Framework.Translators import Convert, Elide, Hole, Inline, Pairwise, Pattern, Prelude, Rule, holes, renames
from mbse.Expressions.Translators import Basic_Python
from mbse.Schemas.Framework import Proxies, Schemas as S
from support import raises, text

BASIC, PYTHON, MATLAB, EXCEL, LATEX = E.DIALECT, P.DIALECT, M.DIALECT, X.DIALECT, L.DIALECT
DIALECTS = [BASIC, PYTHON, MATLAB, EXCEL, LATEX]
EVALUATE = {BASIC: V.OfAny, PYTHON: PV.OfAny, MATLAB: MV.OfAny, EXCEL: XV.OfAny}
NUMPY = Basic_Python.NUMPY
this = E.variable("this")
age = this.age
core = age.ge(18).and_(age.lt(65).or_(this.has("email").not_())).and_(this.name.ne("x")).or_(
    age.add(1).sub(2).mul(3).neg().gt(0).and_(age.eq(1).not_())).and_(this.has("name").eq(True)).and_(
    E.literal(1.5).le(2.5))  # every translatable operation and literal type, operands of one type, no let
rule = E.let_("a", age, E.variable("a").ge(18).implies(this.has("email")))  # a let and implies

Person = S.OfObject.Builder().ref().properties(text("name"), text("age", int), text("email")).create()
Proxies.register("Person", Person)
B = Proxies.Builders.Person
people = [B().name("Ann").age(30).email("a@x").create(), B().name("Bob").age(70).create(),
          B().name("x").age(10).email("c@x").create()]

## TRN-01 · Rules pair patterns; holes bind arguments and attributes

In [ ]:
A, B_, K = *holes("A", "B"), Hole("K", str)
double = Pattern("operation", A, A, name="add")
assert double.size() == 2 and repr(double) == "operation(A, A, name='add')" and double.holes() == {"A": "argument"}
has = Pattern("operation", A, Pattern("literal", value=K), name="has")
assert has.size() == 3 and has.holes() == {"A": "argument", "K": "attribute"}
with raises(ValueError, match="operation(A, B, name='f') needs the argument hole B, which operation(A, name='f') does not bind"):
    Rule(Pattern("operation", A, name="f"), Pattern("operation", A, B_, name="f"))
with raises(ValueError, match="variable(name=A) needs the attribute hole A"):
    Rule(Pattern("operation", A, name="f"), Pattern("variable", name=A))
Rule(Pattern("operation", A, name="f"), Pattern("operation", A, B_, name="f"), "backward")  # only the other way needs B
with raises(ValueError, match="direction must be 'both', 'forward' or 'backward', got 'up'"):
    Rule(double, double, "up")
for declare in [lambda: Inline("let", "up"), lambda: Elide("import", "up"), lambda: Prelude(Pattern("import", A), "up")]:
    with raises(ValueError, match="side must be 'left' or 'right', got 'up'"):
        declare()
for pattern in [Pattern("import", module="numpy"), Pattern("import", Pattern("name", name="x")),
                Pattern("import", A, module=Hole("M"))]:
    with raises(ValueError, match="a prelude is an import whose one argument is a hole"):
        Prelude(pattern, "right")
assert [(r.left.attributes, r.right.attributes) for r in renames("operation", "name", "call", "function",
                                                                  {"eq": "equal"}, 2)] == [
    ({"name": "eq"}, {"function": "equal"})]
# A conversion translates literals by functions of their attributes, which give None where there is no counterpart.
double_ints = Convert("literal", "literal", lambda a: {"value": a["value"] * 2} if type(a["value"]) is int else None,
                      lambda a: {"value": a["value"] // 2})
doubling = Pairwise(BASIC, BASIC, [double_ints])
assert doubling.forward(E.literal(3)).value == 6 and doubling.backward(E.literal(6)).value == 3
assert doubling.inverse().forward(E.literal(6)).value == 3 and doubling.inverse().backward(E.literal(3)).value == 6
with raises(ValueError, match="Basic literal 'a' has no Basic counterpart"):
    doubling.forward(E.literal("a"))
for left, right, named in [("operation", "literal", "Basic 'operation'"), ("literal", "nothing", "Basic 'nothing'")]:
    with raises(ValueError, match=f"a conversion translates literals, and {named} is not one"):
        Pairwise(BASIC, BASIC, [Convert(left, right, double_ints.forward, double_ints.backward)])

# A translator within one dialect is a rewriting: holes bound twice must bind the same node, or equal values.
simplify = Pairwise(BASIC, BASIC, [
    Rule(double, Pattern("operation", A, Pattern("literal", value=2), name="mul"), "forward"),
    Rule(Pattern("operation", Pattern("literal", value=K), Pattern("literal", value=K), name="eq"), Pattern("literal", value=True), "forward"),
    Rule(Pattern("operation", A, B_, name="add"), Pattern("operation", A, B_, name="add"), "forward"),
    Rule(Pattern("operation", Pattern("literal", value=Hole("V")), Pattern("literal", value=Hole("W")), name="eq"),
         Pattern("literal", value=False), "forward"),
    Rule(Pattern("variable", name=Hole("N")), Pattern("variable", name=Hole("N"))),
    Rule(Pattern("literal", value=Hole("V")), Pattern("literal", value=Hole("V")), "forward"),
])
x, y = E.variable("x"), E.variable("y")
assert F.same(simplify.forward(x.add(x)), x.mul(2).data) and F.same(simplify.forward(x.add(y)), x.add(y).data)
assert simplify.forward(E.literal("a").eq("a")).value is True and simplify.forward(E.literal("a").eq("b")).value is False
assert simplify.forward(E.literal(1).eq(1)).value is False  # K holds only a str
assert repr(simplify) == "<translator Basic <-> Basic>"

## TRN-02 · Every pair translates both ways; forms round-trip where both dialects have them

In [ ]:
assert len(Translators.TRANSLATORS) == 12  # every pair of the five, and Basic with Ccpp and SystemVerilog
for source, target in itertools.permutations(DIALECTS, 2):
    translator = Translators.between(source, target)
    assert translator.left() is source and translator.right() is target
    start = Translators.between(BASIC, source).forward(core) if source is not BASIC else core.data
    there = translator.forward(start)
    assert there.dialect() is target and there.validate(bound={"this"}, core=True) == [], (source.name(), target.name())
    assert F.same(translator.backward(there), start), (source.name(), target.name())  # round trip
    assert F.same(translator.inverse().backward(start), there)
with raises(LookupError, match="no translator between Basic and Basic"):
    Translators.between(BASIC, BASIC)

assert P.render(Translators.between(BASIC, PYTHON).forward(rule)) == (
    "(lambda a: hasattr(this, 'email') if a >= 18 else True)(this.age)")
assert M.render(Translators.between(BASIC, MATLAB).forward(rule)) == '~(this.age >= 18) || isfield(this, "email")'
assert X.render(Translators.between(BASIC, EXCEL).forward(rule)) == (
    "=LET(a, this.age, IF(a >= 18, NOT(ISERROR(this.email)), TRUE))")
assert L.render(Translators.between(BASIC, LATEX).forward(rule)) == (
    "a \\geq 18 \\implies \\operatorname{has}(\\mathit{this}, \\text{email}) \\quad \\text{where } a = \\mathit{this}.\\mathit{age}")
assert L.render(Translators.between(PYTHON, LATEX).forward(P.binop("/", P.name("x"), 2))) == "\\frac{x}{2}"
# Python, Excel and Latex have lets and implication, so the rule round-trips; Matlab inlines the let and writes implies
# with ||.
for other in (PYTHON, EXCEL, LATEX):
    assert F.same(Translators.between(other, BASIC).forward(Translators.between(BASIC, other).forward(rule)), rule.data)
back = Translators.between(MATLAB, BASIC).forward(Translators.between(BASIC, MATLAB).forward(rule))
assert F.same(back, age.ge(18).not_().or_(this.has("email")).data)


# NumPy style is a second translator between Basic and Python: numpy's functions, over columns, importing numpy.
styled = NUMPY.forward(rule)
assert P.render(styled) == (
    "import numpy as np\n(lambda a: np.where(np.greater_equal(a, 18), "
    "np.logical_not(np.ma.getmaskarray(this['email'])), True))(this['age'])")
assert F.same(NUMPY.backward(styled), rule.data) and F.same(NUMPY.backward(NUMPY.forward(core)), core.data)
assert P.render(NUMPY.forward(E.literal(1).add(2))) == "import numpy as np\nnp.add(1, 2)"
assert P.render(NUMPY.forward(E.literal(1))) == "1" and P.render(NUMPY.forward(this)) == "this"  # no np, no import
assert F.same(NUMPY.inverse().forward(styled), rule.data) and NUMPY.inverse().backward(E.literal(1)).value == 1
imported = P.import_("math", P.importfrom("numpy", "pi", P.compare(">=", P.attribute(P.name("this"), "age"), 18)))
# imports the other side cannot declare are dropped
for target in (BASIC, MATLAB, EXCEL):
    assert F.same(Translators.between(PYTHON, target).forward(imported),
                  Translators.between(BASIC, target).forward(age.ge(18)) if target is not BASIC else age.ge(18).data)
assert X.render(Translators.between(MATLAB, EXCEL).forward(M.import_("pkg.*", M.binary(">", 1, 0)))) == "=1 > 0"
assert F.same(Translators.between(MATLAB, BASIC).forward(M.import_("pkg.*", M.constant(1))), E.literal(1).data)
with raises(ValueError, match="an import needs a body"):
    Translators.between(PYTHON, MATLAB).forward(P.DIALECT.kinds()["import"]("math"))
with raises(ValueError, match="Python call has no Basic counterpart"):
    Translators.between(PYTHON, BASIC).forward(P.import_("numpy", P.call("np.add", 1, 2), alias="np"))  # read by NUMPY

## TRN-03 · Translating the same pair of dialects by any route gives the same expression

In [ ]:
for a, b, c in itertools.permutations(DIALECTS, 3):
    start = Translators.between(BASIC, a).forward(core) if a is not BASIC else core.data
    direct = Translators.between(a, c).forward(start)
    routed = Translators.between(b, c).forward(Translators.between(a, b).forward(start))
    assert F.same(direct, routed), (a.name(), b.name(), c.name())

## TRN-04 · Co-traversal translates each node once, keeps sharing, and traces the correspondence

In [ ]:
shared = E.operation("add", age, age).data
trace = []
out = Translators.between(BASIC, EXCEL).forward(shared, trace)
assert out.left is out.right  # the shared argument is translated once
assert [(type(s).KIND, type(t).KIND) for s, t in trace] == [("variable", "name"), ("operation", "field"),
                                                             ("operation", "infix")]
assert trace[-1] == (shared, out) and trace[1][1] is out.left
a = E.variable("a")
inlined = Translators.between(BASIC, MATLAB).forward(E.let_("a", age, a.add(a)), trace := [])
assert M.render(inlined) == "this.age + this.age" and inlined.left is inlined.right
assert [type(s).KIND for s, _ in trace] == ["variable", "operation", "variable", "operation", "let"]
shadowed = E.let_("a", 1, E.variable("a").add(E.let_("a", 2, E.variable("a"))))  # the innermost binding wins
assert M.render(Translators.between(BASIC, MATLAB).forward(shadowed)) == "1 + 2"
assert MV.OfAny(Translators.between(BASIC, MATLAB).forward(shadowed)) == V.OfAny(shadowed) == 3
assert M.render(Translators.between(EXCEL, MATLAB).forward(X.let_("n", 2, X.infix("*", X.name("n"), X.name("n"))))) == "2 .* 2"
loop = E.OfOperation.Builder().name("not").create()
E.OfOperation.Builder(loop).arguments(loop).update()
with raises(ValueError, match="the expression contains a cycle"):
    Translators.between(BASIC, MATLAB).forward(loop)
with raises(TypeError, match="not an expression: 7"):
    Translators.between(BASIC, MATLAB).forward(E.OfOperation.Data("not", (7,)))
with raises(ValueError, match="a let needs a value"):
    Translators.between(BASIC, MATLAB).forward(E.OfLet.Data("a"))
with raises(ValueError, match="a let needs a body"):
    Translators.between(EXCEL, MATLAB).forward(X.DIALECT.kinds()["let"]("a", X.constant(1)))

## TRN-05 · What a dialect cannot say is refused, naming the node

In [ ]:
for translate, expression, message in [
    (Translators.between(BASIC, EXCEL).forward, E.literal(b"\x00"), "Basic literal b'\\x00' has no Excel counterpart"),
    (Translators.between(BASIC, PYTHON).forward, E.OfLiteral.Data(), "Basic literal None has no Python counterpart"),
    (Translators.between(BASIC, MATLAB).forward, E.operation("in", 1, 2), "Basic operation 'in' has no Matlab counterpart"),
    (Translators.between(BASIC, PYTHON).forward, this.get(E.variable("key")), "Basic operation 'get' has no Python counterpart"),
    (NUMPY.forward, E.OfOperation.Data("has", (this.data, 7)), "Basic operation 'has'"),
    (Translators.between(BASIC, PYTHON).forward, E.operation("eq", 1, 2, 3), "Basic operation 'eq' has no Python counterpart"),
    (Translators.between(EXCEL, BASIC).forward, X.function("ISERROR", 1), "Excel function 'ISERROR' has no Basic counterpart"),
    (Translators.between(EXCEL, BASIC).forward, X.function("IF", True, 1, 2), "Excel function 'IF' has no Basic counterpart"),
    (Translators.between(EXCEL, BASIC).forward, X.DIALECT.kinds()["let"]("a", X.constant(1)), "Excel let has no Basic counterpart"),
    (Translators.between(PYTHON, MATLAB).forward, P.ifexp(True, 1, 2), "Python ifexp has no Matlab counterpart"),
    (Translators.between(EXCEL, MATLAB).forward, X.cell("A1"), "Excel cell has no Matlab counterpart"),
    (Translators.between(BASIC, LATEX).forward, E.literal(b"\x00"), "Basic literal b'\\x00' has no Latex counterpart"),
    (Translators.between(LATEX, BASIC).forward, L.frac(1, 2), "Latex frac has no Basic counterpart"),
    (Translators.between(PYTHON, LATEX).forward, P.ifexp(True, 1, 2), "Python ifexp has no Latex counterpart"),
    (Translators.between(MATLAB, BASIC).forward, M.call("isfield", M.identifier("s"), M.identifier("k")), "Matlab call 'isfield'"),
]:
    with raises(ValueError, match=message):
        translate(expression)

## TRN-06 · Translations evaluate alike where the dialects' rules agree, and apart where they do not

In [ ]:
EVALUATED = [d for d in DIALECTS if d in EVALUATE]  # Latex has no evaluator
translations = {d: Translators.between(BASIC, d).forward(core) if d is not BASIC else core.data for d in EVALUATED}
lets = {d: Translators.between(BASIC, d).forward(rule) if d is not BASIC else rule.data for d in EVALUATED}
for person in people:
    for expressions in (translations, lets):
        values = {d.name(): bool(EVALUATE[d](e, {"this": person})) for d, e in expressions.items()}
        assert len(set(values.values())) == 1, (person.name, values)
columns = {"age": np.array([30, 70, 10]), "name": np.array(["Ann", "Bob", "x"]),
           "email": np.ma.array(["a@x", "", "c@x"], mask=[False, True, False])}
with_numpy = PV.Scope({"this": columns}, modules={"numpy": np})
assert PV.OfAny(NUMPY.forward(core), with_numpy).tolist() == [
    bool(V.OfAny(core, {"this": person})) for person in people]  # numpy evaluates all three at once

nobody = B().name("Nobody").create()  # no age: each dialect's own notion of a missing value
missing = age.ge(18)
assert V.OfAny(missing, {"this": nobody}) is None  # Basic: unknown
absent = PV.Scope({"this": {"age": np.ma.array([0], mask=[True])}}, modules={"numpy": np})
assert PV.OfAny(NUMPY.forward(missing), absent).mask.tolist() == [True]  # NumPy: masked
with raises(AttributeError, match="'Person' object has no attribute 'age'"):  # Python raises
    PV.OfAny(Translators.between(BASIC, PYTHON).forward(missing), {"this": nobody})
assert XV.OfAny(Translators.between(BASIC, EXCEL).forward(missing), {"this": nobody}) == XV.Error("#FIELD!")  # an error value
with raises(KeyError, match='Unrecognized field name "age".'):  # MATLAB raises
    MV.OfAny(Translators.between(BASIC, MATLAB).forward(missing), {"this": nobody})
mixed, cased = E.literal(1).eq(1.0), E.literal("a").eq("A")
assert [V.OfAny(mixed), V.OfAny(cased)] == [None, False]  # Basic never coerces, and compares text exactly
assert [bool(EVALUATE[d](Translators.between(BASIC, d).forward(mixed))) for d in EVALUATED[1:]] == [True, True, True]
assert [bool(EVALUATE[d](Translators.between(BASIC, d).forward(cased))) for d in EVALUATED[1:]] == [False, False, True]

## TRN-07 · The bitwise operations translate between Basic, Python, MATLAB and Excel, and evaluate alike on non-negative integers

In [ ]:
x = E.variable("x")
bits = x.bitand(12).bitor(x.shr(2)).bitxor(x.shl(1))
RENDER = {PYTHON: P.render, MATLAB: M.render, EXCEL: X.render}
expected = {PYTHON: "(x & 12 | x >> 2) ^ x << 1", MATLAB: "bitxor(bitor(bitand(x, 12), bitshift(x, -2)), bitshift(x, 1))",
            EXCEL: "=BITXOR(BITOR(BITAND(x, 12), BITRSHIFT(x, 2)), BITLSHIFT(x, 1))"}
for dialect, source in expected.items():
    translator = Translators.between(BASIC, dialect)
    translated = translator.forward(bits)
    assert RENDER[dialect](translated) == source and F.same(translator.inverse().forward(translated), bits.data)
    assert EVALUATE[dialect](translated, {"x": 13}) == V.OfAny(bits, {"x": 13}) == 21
# Between the other dialects too, by any route.
for a, b in [(PYTHON, MATLAB), (PYTHON, EXCEL), (MATLAB, EXCEL)]:
    source = Translators.between(BASIC, a).forward(bits)
    target = Translators.between(a, b).forward(source)
    assert F.same(target, Translators.between(BASIC, b).forward(bits)) and F.same(Translators.between(b, a).forward(target), source)
# bitnot is Python's ~ and NumPy's invert; MATLAB and Excel have none for their numbers.
assert P.render(Translators.between(BASIC, PYTHON).forward(x.bitnot())) == "~x"
assert P.render(NUMPY.forward(x.bitnot().shl(1))) == "import numpy as np\nnp.left_shift(np.invert(x), 1)"
with raises(ValueError, match="Basic operation 'bitnot' has no Matlab counterpart"):
    Translators.between(BASIC, MATLAB).forward(x.bitnot())
with raises(ValueError, match="Basic operation 'bitnot' has no Excel counterpart"):
    Translators.between(BASIC, EXCEL).forward(x.bitnot())
with raises(ValueError, match="Python unaryop '~' has no Matlab counterpart"):
    Translators.between(PYTHON, MATLAB).forward(P.parse("~x"))
# Basic's and Python's integers are unbounded and signed; MATLAB's and Excel's bit functions are not.
mask = x.bitand(1)
assert V.OfAny(mask, {"x": -1}) == 1 and PV.OfAny(Translators.between(BASIC, PYTHON).forward(mask), {"x": -1}) == 1
assert XV.OfAny(Translators.between(BASIC, EXCEL).forward(mask), {"x": -1}) == XV.Error("#NUM!")

## TRN-08 · Basic and Ccpp translate into each other; lets are inlined, and `has`, `->` and typed constants have no counterpart

In [ ]:
CCPP = C.DIALECT
x, me = E.variable("x"), E.variable("this")
rule = E.let_("a", me.age, E.variable("a").ge(18).and_(me.width.shl(2).bitand(255).ne(0)).implies(me.name.eq("x")))
translator = Translators.between(BASIC, CCPP)
translated = translator.forward(rule)
assert C.render(translated) == '!(this.age >= 18 && (this.width << 2 & 255) != 0) || this.name == "x"'
expected = E.operation("or", E.operation("not", me.age.ge(18).and_(me.width.shl(2).bitand(255).ne(0))), me.name.eq("x"))
assert F.same(translator.inverse().forward(translated), expected.data)
every = x.add(1).sub(2).mul(3).neg().bitor(x.bitxor(1)).bitand(x.bitnot()).shr(1).lt(x).or_(x.le(1).and_(x.gt(2).not_())).ne(
    x.ge(0).eq(E.literal(True)))
assert F.same(translator.inverse().forward(translator.forward(every)), every.data) and C.render(translator.forward(E.literal(1.5))) == "1.5"
for value in [3, 9]:  # C's ints and Basic's agree where neither overflows
    assert CV.OfAny(translator.forward(x.add(1).mul(2).gt(5).and_(x.lt(9))), {"x": value}) == V.OfAny(x.add(1).mul(2).gt(5).and_(x.lt(9)), {"x": value})
# Typed constants are literals of value domains: C's integer types by width and signedness, its floating types by format.
U8 = D.OfInteger.Data(8, False)
for literal, source in [(E.literal(200, U8), "(uint8_t)200"), (E.literal(-5, D.OfInteger.Data(64)), "(int64_t)-5"),
                        (E.literal(1.5, D.OfIeee754.Data("binary32")), "1.5f"), (E.literal("0.1", D.OfIeee754.Data("binary128")), "0.1L")]:
    there = translator.forward(literal)
    assert C.render(there) == source and F.same(translator.inverse().forward(there), literal.data), source
for constant, domain in [(C.constant(5, "unsigned int"), D.OfInteger.Data(32, False)), (C.constant(5, "size_t"), D.OfInteger.Data(64, False))]:
    assert translator.inverse().forward(constant).domain == domain
for constant, plain in [(C.constant(True, "bool"), True), (C.constant(1.5, "double"), 1.5)]:  # binary64 is a float's own domain
    assert F.same(translator.inverse().forward(constant), E.literal(plain).data)
assert CV.OfAny(translator.forward(E.literal(200, U8).add(E.literal(100, U8)))) == 300  # in C's int, where Basic's uint8 overflows
for expression, source, message in [
    (me.has("email"), BASIC, "Basic operation 'has' has no Ccpp counterpart"),
    (E.literal(200, D.OfInteger.Data(12)), BASIC, "Basic literal 200 of int12 has no Ccpp counterpart"),
    (E.literal(b"\x01", D.OfBits.Data(8)), BASIC, "Basic literal b'\\x01' of bits8 has no Ccpp counterpart"),
    (C.constant(5, "word"), CCPP, "Ccpp constant 5 has no Basic counterpart"),
    (C.member(C.identifier("x"), "a", "->"), CCPP, "Ccpp member has no Basic counterpart"),
]:
    with raises(ValueError, match=message):
        Translators.between(source, BASIC if source is CCPP else CCPP).forward(expression)

## TRN-09 · Basic and SystemVerilog translate into each other; lets are inlined, bools are single bits, and `has`, sized vectors and 4-state comparisons have no counterpart

In [ ]:
SYSTEMVERILOG = SV.DIALECT
x, me = E.variable("x"), E.variable("this")
rule = E.let_("a", me.age, E.variable("a").ge(18).and_(me.width.shr(2).bitand(255).ne(0)).implies(me.name.eq("x")))
translator = Translators.between(BASIC, SYSTEMVERILOG)
translated = translator.forward(rule)
assert SV.render(translated) == 'this.age >= 18 && (this.width >>> 2 & 255) != 0 -> this.name == "x"'
expected = me.age.ge(18).and_(me.width.shr(2).bitand(255).ne(0)).implies(me.name.eq("x"))
assert F.same(translator.inverse().forward(translated), expected.data)
every = x.add(1).sub(2).mul(3).neg().bitor(x.bitxor(1)).bitand(x.bitnot()).shl(1).lt(x).or_(x.le(1).and_(x.gt(2).not_())).ne(
    x.ge(0).eq(E.literal(True)).or_(E.literal(False)))
assert F.same(translator.inverse().forward(translator.forward(every)), every.data)
assert SV.render(translator.forward(E.literal(True).and_(E.literal(1.5).lt(2)))) == "1'b1 && 1.5 < 2"
for value in [3, 9]:  # SystemVerilog's integers and Basic's agree where neither overflows
    check = x.add(1).mul(2).gt(5).and_(x.lt(9))
    assert SVV.OfAny(translator.forward(check), {"x": value}).integer() == int(V.OfAny(check, {"x": value}))
# Sized vectors are literals of value domains, by their base: decimal integers, hexadecimal bits, single std_logic bits.
U8, LOGIC = D.OfInteger.Data(8, False), D.OfIeee1164.Data()
for literal, source in [(E.literal(200, U8), "8'd200"), (E.literal(-5, D.OfInteger.Data(8)), "8'sd251"), (E.literal(5, D.OfInteger.Data(12)), "12'sd5"),
                        (E.literal(b"\x0a\xbc", D.OfBits.Data(12)), "12'habc"), (E.literal("X", LOGIC), "1'bx"), (E.literal("1", LOGIC), "1'b1")]:
    there = translator.forward(literal)
    assert SV.render(there) == source and F.same(translator.inverse().forward(there), literal.data), source
for vector, value, domain in [(SV.vector("1010"), b"\x0a", D.OfBits.Data(4)), (SV.vector("1111", True, "h"), -1, D.OfInteger.Data(4)),
                              (SV.vector("101", base="o"), b"\x05", D.OfBits.Data(3)), (SV.vector("z"), "Z", LOGIC), (SV.vector("0", base="b"), "0", LOGIC)]:
    literal = translator.inverse().forward(vector)
    assert literal.value == value and literal.domain == domain, SV.render(vector)
assert repr(SVV.OfAny(translator.forward(E.literal(200, U8).add(E.literal(100, U8))))) == "8'b00101100"  # wraps in 8 bits, where Basic's overflows
for expression, source, message in [
    (me.has("email"), BASIC, "Basic operation 'has' has no SystemVerilog counterpart"),
    (E.literal(1.5, D.OfIeee754.Data("binary32")), BASIC, "Basic literal 1.5 of binary32 has no SystemVerilog counterpart"),
    (E.literal("U", D.OfIeee1164.Data()), BASIC, "Basic literal 'U' of std_logic has no SystemVerilog counterpart"),
    (SV.vector("10x1"), SYSTEMVERILOG, "SystemVerilog vector '10x1' of logic [3:0] has no Basic counterpart"),
    (SV.binary("===", SV.identifier("x"), 1), SYSTEMVERILOG, "SystemVerilog binary '===' has no Basic counterpart"),
]:
    with raises(ValueError, match=message):
        Translators.between(source, BASIC if source is SYSTEMVERILOG else SYSTEMVERILOG).forward(expression)